# Circuits in DeepLog

A [Circuit](deeplog.circuit.circuit.Circuit) is a DAG over a single algebraic structure. After construction, `to_module()` bundles the circuit into a [`DeepLogModule`](deeplog.module.deeplog_module.DeepLogModule); when predicates are registered, the circuit will also materialize their modules automatically.

In [ ]:
import torch

from deeplog.circuit import Circuit


# Construct circuit
circuit = Circuit("boolean")
symbol_a = ("a",)
symbol_b = ("b",)
symbol_c = ("c",)
leaf_a = circuit.get_leaf_node(symbol_a)
leaf_b = circuit.get_leaf_node(symbol_b)
leaf_c = circuit.get_leaf_node(symbol_c)

and_operator = circuit.get_operator("and")
or_operator = circuit.get_operator("or")
and_node = and_operator(leaf_a, leaf_b)
or_node = or_operator(leaf_c, and_node)

# Use circuit
module = circuit.to_module({or_node: ("or_root",), and_node: ("and_root",)})
print("Module created:", module)

inputs = torch.tensor([[1, 0, 1]], dtype=torch.float32)  # Example input: a=1, b=0, c=1
output = module(inputs)
print(output)

Observe in the example: each leaf node has an associated unique, symbolic name (cf. `circuit.get_leaf_node(_)`). To combine these leaf nodes, we first obtain from the circuit a reference to the desired operator (cf. `circuit.get_operator(_)`). The available operators are defined by the algebraic structure. For `boolean` this is `and`, `or`, and `not`, while for `probability` this is `times`, `plus`, and `negate`.


In [ ]:
from deeplog import BOOLEAN, PROBABILITY, LOGPROBABILITY


print("Boolean operators:", BOOLEAN.operators)
print("Probability operators:", PROBABILITY.operators)
print("LogProbability operators:", LOGPROBABILITY.operators)

As we combine nodes we construct a computational graph. We then set the root nodes to indicate the nodes we are interested in, before converting the circuit into an evaluatable [DeepLogModule](deeplog.module.deeplog_module.DeepLogModule).

Next we show a small example for the `"probability"` structure.

In [ ]:
import torch

from deeplog.circuit import Circuit


# Construct circuit
circuit = Circuit("probability")  # <--- probability
symbol_a = ("a",)
symbol_b = ("b",)
leaf_a = circuit.get_leaf_node(symbol_a)
leaf_b = circuit.get_leaf_node(symbol_b)
or_operator = circuit.get_operator("plus")  # <--- plus
or_node = or_operator(leaf_a, leaf_b)

# Use circuit
module = circuit.to_module({or_node: ("or_root",)})
print("Module created:", module)

inputs = torch.tensor(
    [[0.4, 0.25]], dtype=torch.float32
)  # Example input: a=0.4, b=0.25
output = module(inputs)
print(output)

## Deterministic Circuits
The `probability` structure uses `plus`/`times` as semiring operators (not literal boolean OR), so adding two probabilities is only a true union if the events are disjoint. For deterministic circuits that enforce this property, pass `deterministic=True` to the `Circuit` constructor, which compiles via PySDD.

In [ ]:
import torch

from deeplog.circuit import Circuit


# Construct circuit
circuit = Circuit("probability", deterministic=True)  # <--- deterministic
symbol_a = ("a",)
symbol_b = ("b",)
leaf_a = circuit.get_leaf_node(symbol_a)
leaf_b = circuit.get_leaf_node(symbol_b)
or_operator = circuit.get_operator("plus")
or_node = or_operator(leaf_a, leaf_b)

# Use circuit
module = circuit.to_module({or_node: ("or_root",)})
print("Module created:", module)

inputs = torch.tensor(
    [[0.4, 0.25]], dtype=torch.float32
)  # Example input: a=0.4, b=0.25
output = module(inputs)
print(output)  # 0.25 + (1 - 0.25) * 0.4 = 0.55
